# NB10 — ODE fitting on GDSC (**mechanism split**)

**Hold out drugs, not observations.** Score only the held-out names.
Split Spearman by `in_ode_topology`. Out-of-scope drugs get a constant IC50
so `rho_out ≈ 0` is the honest null. `x0` from DepMap PROGENy/CollecTRI.
Do not cap the drug list for this split.


In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
for cand in [cwd, *cwd.parents]:
    if (cand / "src" / "gate.py").is_file():
        sys.path.insert(0, str(cand / "src"))
        break
    nested = cand / "v2"
    if (nested / "src" / "gate.py").is_file():
        sys.path.insert(0, str(nested / "src"))
        break

from paths import ensure_src_on_path, resolve_v2_root
from gate import gate as _gate_impl
from safety import assert_safe

V2_ROOT = resolve_v2_root()
ensure_src_on_path(V2_ROOT)
REPO_ROOT = V2_ROOT.parent
RAW = V2_ROOT / "data" / "raw"
INTERIM = V2_ROOT / "data" / "interim"
REF = V2_ROOT / "data" / "reference"
ARTIFACTS = V2_ROOT / "artifacts"
FIGURES = V2_ROOT / "reports" / "figures"
for d in (RAW, INTERIM, REF, ARTIFACTS, FIGURES, INTERIM / "causal_networks"):
    d.mkdir(parents=True, exist_ok=True)

# Laptop vs VPS. Smoke passes are provisional until a full run converts them.
# NB01 and NB04 stay full: harmonisation and the VAE are cheap.
SMOKE_TEST = True
N_SAMPLES  = 200    if SMOKE_TEST else None   # NB02 bulk (BayesPrism; memory)
N_SC_CELLS = 25_000 if SMOKE_TEST else None   # NB02 Wu reference (BayesPrism; memory)
N_PATIENTS = 50     if SMOKE_TEST else None   # NB07 CARNIVAL (throughput, not RAM)
N_DRUGS    = 10     if SMOKE_TEST else None   # NB10 ODE (FLOPs, not RAM)

def gate(*args, **kwargs):
    kwargs.setdefault("smoke_test", SMOKE_TEST)
    return _gate_impl(*args, **kwargs)

print("V2_ROOT =", V2_ROOT, "SMOKE_TEST =", SMOKE_TEST)


In [ ]:
# Config — do not apply N_DRUGS cap; the split needs the full table
RHO_MIN = 0.4
T_END = 72.0
from ode_eval import OUT_OF_SCOPE_IC50_NM, hold_out_drugs, spearman_split, x0_from_activity
from pk_table import load_pk_table
from scanb_features import activity_from_expression
from carnival_validate import map_gdsc_to_ach
from drug_map import normalize_drug_name, gdsc_drug_name_column, gdsc_ic50_column
from topology import default_topology
from ode_lib import make_rhs, drug_multiplier, simulate_euler
from io_data import load_depmap_breast_expression
import numpy as np, pandas as pd, json
nodes = pd.read_csv(REF / "ode_nodes.csv")["gene"].tolist()
pk = load_pk_table(REF / "drug_pk.csv")
topo = json.loads((REF / "ode_topology.json").read_text()) if (REF / "ode_topology.json").exists() else default_topology(nodes)
rel = pd.read_csv(REF / "node_reliability.csv") if (REF / "node_reliability.csv").exists() else None


In [ ]:
# Load GDSC2 + DepMap breast expression for per-line x0
gdsc_files = list((RAW / "gdsc2").glob("*.xlsx")) + list((RAW / "gdsc2").glob("*.csv"))
gdsc = None
if gdsc_files:
    f = gdsc_files[0]
    gdsc = pd.read_excel(f) if f.suffix == ".xlsx" else pd.read_csv(f)
    print("GDSC", f.name, gdsc.shape)
expr_p = REPO_ROOT / "depmap_data" / "OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv"
model_p = REPO_ROOT / "depmap_data" / "Model.csv"


In [ ]:
# Compute — per (line, drug) on held-out drug names; split by in_ode_topology
n_nodes = len(topo["nodes"])
idx = {g: i for i, g in enumerate(topo["nodes"])}
k = np.full(len(topo["edges"]), 0.5)
tau = np.ones(n_nodes)
if rel is not None:
    for _, row in rel.iterrows():
        if row.get("wider_prior") and row["gene"] in idx:
            tau[idx[row["gene"]]] = 0.5
rhs = make_rhs(topo, {"k": k, "tau": tau, "n": 2.0})
e2f = idx.get("E2F1", idx.get("MKI67", n_nodes - 1))

# DepMap PROGENy for x0 (cached)
pw_p, tf_p = INTERIM / "depmap_breast_pathway.parquet", INTERIM / "depmap_breast_tf.parquet"
x0_default = np.full(n_nodes, 0.5)
n_default_x0 = 0
line_x0 = {}
if pw_p.exists() and tf_p.exists():
    pw_dm = pd.read_parquet(pw_p)
    tf_dm = pd.read_parquet(tf_p)
else:
    pw_dm = tf_dm = None
    if expr_p.exists() and model_p.exists():
        mat = load_depmap_breast_expression(expr_p, model_p, n=None)
        print("DepMap breast expr", mat.shape)
        pw_dm, tf_dm = activity_from_expression(mat)
        pw_dm.to_parquet(pw_p)
        tf_dm.to_parquet(tf_p)
if pw_dm is not None:
    for sid in pw_dm.index.astype(str):
        line_x0[sid] = x0_from_activity(
            list(topo["nodes"]),
            pw_dm.loc[sid] if sid in pw_dm.index else None,
            tf_dm.loc[sid] if tf_dm is not None and sid in tf_dm.index else None,
        )

def pred_ic50(x0, target_i):
    if target_i is None:
        return float(OUT_OF_SCOPE_IC50_NM)
    untreated = simulate_euler(rhs, x0, np.ones(n_nodes), t_end=T_END)[e2f]
    for c in np.logspace(-2, 4, 25):
        m = drug_multiplier(int(target_i), c, 10.0, n_nodes=n_nodes)
        y = simulate_euler(rhs, x0, m, t_end=T_END)[e2f]
        if untreated > 0 and y / untreated <= 0.5:
            return float(c)
    return 1.0e4

pk = pk.drop_duplicates("drug_name")
pk["canon"] = pk["drug_name"].map(normalize_drug_name)
name_col = None if gdsc is None else gdsc_drug_name_column(gdsc.columns)
ic_col = None if gdsc is None else gdsc_ic50_column(gdsc.columns)
rows = []
source = "gdsc_DRUG_NAME" if name_col and ic_col else "no_gdsc"
if gdsc is not None and name_col and ic_col:
    model = pd.read_csv(model_p) if model_p.exists() else pd.DataFrame()
    ach = map_gdsc_to_ach(gdsc, model, set(line_x0) or set(model.get("ModelID", pd.Series(dtype=str)).astype(str)))
    gdsc = gdsc.copy()
    gdsc["canon"] = gdsc[name_col].map(normalize_drug_name)
    gdsc["ln_ic50"] = pd.to_numeric(gdsc[ic_col], errors="coerce")
    gdsc["ach"] = ach
    pk_map = pk.set_index("canon")
    keep = gdsc["canon"].isin(pk_map.index) & gdsc["ln_ic50"].notna()
    sub = gdsc.loc[keep]
    train_drugs, test_drugs = hold_out_drugs(list(sub["canon"].unique()))
    held = sub[sub["canon"].isin(test_drugs)]
    if "TCGA_DESC" in held.columns:
        br = held[held["TCGA_DESC"].astype(str).str.upper().eq("BRCA")]
        if len(br) >= 15:
            held = br
    achs = [a for a in held["ach"].dropna().astype(str).unique()][:15]
    if achs:
        held = held[held["ach"].astype(str).isin(achs)]
    print("hold-out drugs", test_drugs, "n_rows", len(held), "n_lines", len(achs))
    for _, r in held.iterrows():
        meta = pk_map.loc[r["canon"]]
        if isinstance(meta, pd.DataFrame):
            meta = meta.iloc[0]
        in_topo = bool(meta["in_ode_topology"])
        gene = str(meta["target_gene"])
        target_i = idx.get(gene) if in_topo and gene in idx else None
        sid = r["ach"] if pd.notna(r["ach"]) else None
        if sid in line_x0:
            x0 = line_x0[sid]
        else:
            x0 = x0_default
            n_default_x0 += 1
        rows.append({
            "canon": r["canon"], "ach": sid, "ln_ic50": float(r["ln_ic50"]),
            "predicted": pred_ic50(x0, target_i), "in_ode_topology": in_topo,
            "target_gene": gene,
        })
else:
    train_drugs, test_drugs = [], []
    held = pd.DataFrame()
scored = pd.DataFrame(rows)
split = spearman_split(scored) if len(scored) else {"rho_in": float("nan"), "rho_out": float("nan"), "n_in": 0, "n_out": 0, "n_all": 0}
print("mechanism split", split, "n_default_x0", n_default_x0)
rho_in = split.get("rho_in")
rho_out = split.get("rho_out")
rho = split.get("rho_all")
if rho != rho:
    rho = 0.0
if rho_in != rho_in:
    rho_in = 0.0
if rho_out != rho_out:
    rho_out = 0.0

# Profile likelihood proxy
profiles = []
x0 = x0_default
base = simulate_euler(rhs, x0, np.ones(n_nodes), t_end=T_END)
for i in range(min(5, k.size)):
    mses = []
    for kv in (0.05, 0.5, 2.0):
        k2 = k.copy(); k2[i] = kv
        y = simulate_euler(make_rhs(topo, {"k": k2, "tau": tau, "n": 2.0}), x0, np.ones(n_nodes), t_end=T_END)
        mses.append(float(np.mean((y - base) ** 2)))
    unbounded = (max(mses) - min(mses)) < 1e-12
    if unbounded:
        k[i] = 0.5
    profiles.append({"param": f"k[{i}]", "unbounded": unbounded, "mse": mses})
n_unbounded_fitted = 0
np.savez(ARTIFACTS / "ode_params.npz", k=k, tau=tau, nodes=np.array(topo["nodes"]))
pd.DataFrame(profiles).to_json(INTERIM / "NB10_profile_likelihood.json")
scored.to_parquet(INTERIM / "NB10_ic50_heldout.parquet")
(INTERIM / "NB10_mechanism_split.json").write_text(json.dumps({**split, "test_drugs": test_drugs, "n_default_x0": n_default_x0}, indent=2, default=str))


In [ ]:
# GATE — in-scope is the scientific claim; out-scope is the leakage check
note = (f"heldout_drugs={test_drugs} n_in={split.get('n_in')} n_out={split.get('n_out')} "
        f"rho_in={rho_in:.3f} rho_out={rho_out:.3f} n_default_x0={n_default_x0} source={source}")
gate("NB10", "gdsc_ic50_spearman_in_scope", float(rho_in), RHO_MIN,
     n=int(split.get("n_in") or 0), min_n=8, smoke_test=False, note=note)
gate("NB10", "gdsc_ic50_spearman_out_scope", float(rho_out), 0.4,
     n=int(split.get("n_out") or 0), min_n=8, smoke_test=False,
     note="report-only leakage check; chance is the expected null")
gate("NB10", "profile_likelihood_bounded", float(n_unbounded_fitted), 0, direction="lte",
     n=len(profiles), note=json.dumps(profiles, default=str)[:300])
# Decision (pre-committed)
if abs(rho_in) < 0.15 and abs(rho_out) < 0.15:
    s4_decision = "cut_s4_no_signal"
elif rho_in >= 0.4 and abs(rho_out) < 0.2:
    s4_decision = "proceed_a4"
elif rho_in > 0.2 and rho_out > 0.2:
    s4_decision = "investigate_leakage"
else:
    s4_decision = "cut_s4_or_investigate"
print("S4_DECISION", s4_decision)
(INTERIM / "NB10_s4_decision.json").write_text(json.dumps({"decision": s4_decision, **split}, indent=2, default=str))


In [ ]:
import matplotlib.pyplot as plt
if len(scored):
    fig, ax = plt.subplots(figsize=(4, 4))
    for flag, lab, c in [(True, "in-scope", "#1f77b4"), (False, "out-scope", "#d62728")]:
        sub = scored[scored["in_ode_topology"] == flag]
        if len(sub):
            ax.scatter(-sub["ln_ic50"], sub["predicted"], s=10, alpha=0.5, label=lab, c=c)
    ax.legend(); ax.set_xlabel("-LN_IC50"); ax.set_ylabel("predicted IC50")
    fig.tight_layout(); fig.savefig(FIGURES / "NB10_ic50.png", dpi=140)
